# Stanford News: collect AI article links

Stanford Report rejects plain `requests` calls with HTTP 403, so this notebook uses Playwright + Chromium to load the AI category page and walk its pagination controls.

Output: `./Data/Stanford Articles Links.csv`


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "playwright": "playwright",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
    "tqdm": "tqdm",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

# Idempotent: an already-installed Chromium build is reused.
subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])


In [ ]:
import random
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlsplit, urlunsplit

import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from tqdm.auto import tqdm


In [ ]:
BASE_URL = "https://news.stanford.edu"
START_URL = f"{BASE_URL}/artificial-intelligence"
OUTPUT_PATH = Path("./Data/Stanford Articles Links.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Set False if Stanford ever blocks headless Chromium on your network.
HEADLESS = False
NAVIGATION_TIMEOUT_MS = 60_000
PAGE_CHANGE_TIMEOUT_MS = 30_000
MAX_PAGES = 100
MIN_DELAY = 0.8
MAX_DELAY = 1.5

ARTICLE_PATH_RE = re.compile(r"^/stories/\d{4}/\d{2}/[^/?#]+/?$")


In [ ]:
def clean_text(value):
    if value is None:
        return ""
    return " ".join(str(value).split())


def canonicalize_url(url):
    parts = urlsplit(url)
    path = parts.path.rstrip("/") or "/"
    return urlunsplit((parts.scheme.lower(), parts.netloc.lower(), path, "", ""))


def is_stanford_story(url):
    parts = urlsplit(url)
    return (
        parts.scheme in {"http", "https"}
        and parts.netloc.lower() == "news.stanford.edu"
        and ARTICLE_PATH_RE.match(parts.path) is not None
    )


def infer_title(anchor):
    # Prefer a nearby card heading; otherwise use the anchor text itself.
    for parent in list(anchor.parents)[:6]:
        if getattr(parent, "name", None) in {"h2", "h3", "h4"}:
            title = clean_text(parent.get_text(" ", strip=True))
            if title:
                return title

        if getattr(parent, "name", None) in {"article", "li", "div"}:
            heading = parent.find(["h2", "h3", "h4"])
            if heading:
                title = clean_text(heading.get_text(" ", strip=True))
                if title:
                    return title

    return clean_text(anchor.get_text(" ", strip=True))


def extract_story_records(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    root = soup.find("main") or soup
    records = {}

    for anchor in root.find_all("a", href=True):
        absolute_url = canonicalize_url(urljoin(page_url, anchor["href"]))
        if not is_stanford_story(absolute_url):
            continue

        title = infer_title(anchor)
        if not title or title.casefold() in {"read more", "learn more", "view story"}:
            continue

        previous = records.get(absolute_url, "")
        if len(title) > len(previous):
            records[absolute_url] = title

    return [{"Article": title, "Link": url} for url, title in records.items()]


def story_url_set(records):
    return {row["Link"] for row in records}


In [ ]:
async def dismiss_cookie_banner(page):
    selectors = [
        'button:has-text("Accept")',
        'button[aria-label*="accept" i]',
    ]
    for selector in selectors:
        locator = page.locator(selector).first
        try:
            if await locator.is_visible(timeout=1_000):
                await locator.click(timeout=2_000)
                return
        except Exception:
            pass


async def current_records(page):
    return extract_story_records(await page.content(), page.url)


async def wait_for_story_change(page, previous_urls):
    deadline = time.monotonic() + PAGE_CHANGE_TIMEOUT_MS / 1000

    while time.monotonic() < deadline:
        await page.wait_for_timeout(350)
        records = await current_records(page)
        urls = story_url_set(records)
        if urls and urls != previous_urls:
            return records

    return None


async def click_exact_page_number(page, page_number):
    # Restrict the search to links/buttons whose complete visible text is the
    # requested number. This avoids matching article titles containing digits.
    controls = page.locator("main a, main button")
    count = await controls.count()

    for index in range(count):
        control = controls.nth(index)
        try:
            text = clean_text(await control.inner_text(timeout=1_000))
            if text != str(page_number):
                continue
            if not await control.is_visible():
                continue
            if await control.is_disabled():
                continue
            await control.click(timeout=10_000)
            return True
        except Exception:
            continue

    return False


async def click_next_control(page):
    selectors = [
        'main a[rel="next"]',
        'main a[aria-label*="next" i]',
        'main button[aria-label*="next" i]',
        'main a[title*="next" i]',
        'main button[title*="next" i]',
        'main a:has-text("Next")',
        'main button:has-text("Next")',
    ]

    for selector in selectors:
        control = page.locator(selector).first
        try:
            if not await control.is_visible(timeout=800):
                continue
            if await control.is_disabled():
                continue
            await control.click(timeout=10_000)
            return True
        except Exception:
            continue

    return False


In [ ]:
async def collect_stanford_links():
    all_records = []
    page_number = 1

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=HEADLESS)
        context = await browser.new_context(
            locale="en-US",
            viewport={"width": 1440, "height": 1200},
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/126.0.0.0 Safari/537.36"
            ),
        )
        page = await context.new_page()
        page.set_default_navigation_timeout(NAVIGATION_TIMEOUT_MS)
        page.set_default_timeout(30_000)

        response = await page.goto(START_URL, wait_until="domcontentloaded")
        if response is not None and response.status >= 400:
            raise RuntimeError(
                f"Stanford returned HTTP {response.status}. "
                "Set HEADLESS=False and rerun this cell."
            )

        await page.wait_for_selector("main")
        await dismiss_cookie_banner(page)

        progress = tqdm(total=None, desc="Category pages")

        while page_number <= MAX_PAGES:
            records = await current_records(page)
            if not records:
                raise RuntimeError(
                    f"No Stanford story links found on rendered page {page_number}: {page.url}"
                )

            before_urls = story_url_set(records)
            all_records.extend(records)

            progress.update(1)
            progress.set_postfix(
                page=page_number,
                page_articles=len(records),
                unique_articles=len({row["Link"] for row in all_records}),
            )

            next_page_number = page_number + 1
            clicked = await click_exact_page_number(page, next_page_number)
            if not clicked:
                clicked = await click_next_control(page)

            if not clicked:
                break

            changed_records = await wait_for_story_change(page, before_urls)
            if changed_records is None:
                print(
                    f"Pagination control was clicked, but page {next_page_number} "
                    "did not produce a new set of story links. Stopping."
                )
                break

            page_number = next_page_number
            await page.wait_for_timeout(int(random.uniform(MIN_DELAY, MAX_DELAY) * 1000))

        progress.close()
        await context.close()
        await browser.close()

    if page_number >= MAX_PAGES:
        print(f"Stopped at MAX_PAGES={MAX_PAGES}; increase it if necessary.")

    return all_records, page_number


all_records, category_pages_visited = await collect_stanford_links()


In [ ]:
article_links_df = (
    pd.DataFrame(all_records, columns=["Article", "Link"])
    .drop_duplicates(subset="Link")
    .sort_values(["Article", "Link"], kind="stable")
    .reset_index(drop=True)
)

print(f"Category pages visited: {category_pages_visited}")
print(f"Unique Stanford articles: {len(article_links_df)}")
display(article_links_df.head(10))


In [ ]:
article_links_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH.resolve()}")
